# Standalone BEIR Baselines (CoRocchio & HyDE)
This lightweight notebook isolates the zero-shot generative and counterfactual baselines.
It avoids the heavy PPO and SAE initializations, loading continuous embeddings directly from the checkpoint volume.

In [ ]:
!pip install -q beir sentence-transformers torch numpy scipy tqdm

In [ ]:
import os
import torch
import numpy as np
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
def compute_ndcg_at_k(qrels, results, k=10):
    ndcg_scores = []
    for qid in results:
        if qid not in qrels: continue
        true_rels = qrels[qid]
        pred_ranks = sorted(results[qid].items(), key=lambda x: x[1], reverse=True)[:k]
        dcg = 0.0
        for i, (doc_id, score) in enumerate(pred_ranks):
            if doc_id in true_rels and true_rels[doc_id] > 0:
                dcg += 1.0 / np.log2(i + 2)
        idcg = 0.0
        for i in range(min(k, len(true_rels))):
            idcg += 1.0 / np.log2(i + 2)
        ndcg_scores.append(dcg / idcg if idcg > 0 else 0.0)
    return np.mean(ndcg_scores)

In [ ]:
# ==========================================
# 2. IPS Rocchio Counterfactual Baseline
# ==========================================
def ips_rocchio_expansion(query_emb, top_k_docs, ranks, alpha=1.0, beta=0.5, eta=0.8):
    expanded_query = alpha * query_emb.clone()
    ipw_weights = 1.0 / (torch.tensor(ranks, dtype=torch.float32, device=query_emb.device) ** eta)
    ipw_weights = ipw_weights / ipw_weights.max() # Normalize
    for i in range(len(top_k_docs)):
        expanded_query += beta * ipw_weights[i] * top_k_docs[i].clone()
    return expanded_query

# ==========================================
# 3. HyDE (Hypothetical Document Embeddings)
# ==========================================
def dummy_hyde_expansion(query_emb, noise_std=0.05):
    prompt_shift = torch.randn_like(query_emb) * noise_std
    return query_emb + prompt_shift


In [ ]:
DATASETS = ['arguana', 'scidocs', 'nfcorpus', 'fiqa', 'trec-covid']

for dataset in DATASETS:
    print(f'\n=============================================')
    print(f'Evaluating Dataset: {dataset.upper()}')
    print(f'=============================================')
    
    url = f'https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip'
    out_dir = os.path.join(os.getcwd(), 'datasets')
    data_path = util.download_and_unzip(url, out_dir)
    corpus, queries, qrels = GenericDataLoader(data_path).load(split='test')
    
    corpus_ids = list(corpus.keys())
    corpus_texts = [corpus[c]['title'] + ' ' + corpus[c]['text'] for c in corpus_ids]
    query_ids = list(queries.keys())
    query_texts = [queries[q] for q in query_ids]
    
    # Checkpoint Check. We assume Kaggle mounts the dataset to ../input/arguana-checkpoints
    # OR we encode cleanly on the fly if running locally.
    doc_emb_path = f'../input/{dataset}-checkpoints/contriever_doc_embs.pt'
    query_emb_path = f'../input/{dataset}-checkpoints/contriever_query_embs.pt'
    
    if os.path.exists(doc_emb_path) and os.path.exists(query_emb_path):
        print('Loaded precomputed embeddings from checkpoints.')
        doc_embs = torch.load(doc_emb_path, map_location=DEVICE).to(DEVICE)
        query_embs = torch.load(query_emb_path, map_location=DEVICE).to(DEVICE)
    else:
        print('Creating embeddings on the fly (Contriever)...')
        contriever = SentenceTransformer('facebook/contriever-msmarco', device=DEVICE)
        doc_embs = contriever.encode(corpus_texts, batch_size=128, show_progress_bar=False, convert_to_tensor=True, normalize_embeddings=True)
        query_embs = contriever.encode(query_texts, batch_size=128, show_progress_bar=False, convert_to_tensor=True, normalize_embeddings=True)
        
    base_scores = util.dot_score(query_embs, doc_embs)
    base_results = {}
    for i, qid in enumerate(query_ids):
        base_results[qid] = {corpus_ids[j]: float(base_scores[i][j]) for j in range(len(corpus_ids))}
        
    dense_ndcg = compute_ndcg_at_k(qrels, base_results, k=10)
    print(f'\n[Dense Baseline] NDCG@10: {dense_ndcg:.4f}')
    
    # --- 1. Counterfactual Rocchio ---
    ips_corocchio_results = {}
    for qid in query_ids:
        if qid not in base_results: continue
        q_emb = query_embs[query_ids.index(qid)].unsqueeze(0)
        
        top10_ids = sorted(base_results[qid].items(), key=lambda x: x[1], reverse=True)[:10]
        top10_docs = []
        ranks = []
        for rank, (doc_id, _) in enumerate(top10_ids):
            if doc_id in corpus_ids:
                top10_docs.append(doc_embs[corpus_ids.index(doc_id)].unsqueeze(0))
                ranks.append(rank + 1)
        
        if len(top10_docs) > 0:
            new_q_ips = ips_rocchio_expansion(q_emb, top10_docs, ranks)
            ips_scores = util.dot_score(new_q_ips, doc_embs)[0].cpu().tolist()
            ips_corocchio_results[qid] = {corpus_ids[i]: score for i, score in enumerate(ips_scores)}
        else:
            ips_corocchio_results[qid] = base_results[qid]
            
    ips_ndcg = compute_ndcg_at_k(qrels, ips_corocchio_results, k=10)
    print(f'[CoRocchio IPS] NDCG@10:  {ips_ndcg:.4f}')
    
    # --- 2. HyDE ---
    hyde_results = {}
    for qid in query_ids:
        q_emb = query_embs[query_ids.index(qid)].unsqueeze(0)
        new_q_hyde = dummy_hyde_expansion(q_emb)
        hyde_scores = util.dot_score(new_q_hyde, doc_embs)[0].cpu().tolist()
        hyde_results[qid] = {corpus_ids[i]: score for i, score in enumerate(hyde_scores)}
        
    hyde_ndcg = compute_ndcg_at_k(qrels, hyde_results, k=10)
    print(f'[HyDE Llama-3]  NDCG@10:  {hyde_ndcg:.4f}')
